# Part B - Text-Based Product Matching

In [29]:
!pip install unidecode

In [30]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.metrics as metrics
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import (precision_recall_curve, average_precision_score,
                             precision_score, recall_score, f1_score)
import kagglehub
import itertools
from unidecode import unidecode
from collections import defaultdict
from sklearn.model_selection import GroupShuffleSplit
from sentence_transformers import SentenceTransformer
import time

## Data Loading

In [31]:
kagglehub.login()


In [32]:
path = kagglehub.competition_download('shopee-product-matching')

print("Path to competition files:", path)

Path to competition files: /root/.cache/kagglehub/competitions/shopee-product-matching


In [33]:
df=pd.read_csv(os.path.join(path,'train.csv'))
df.head()

,posting_id,image,image_phash,title,label_group
0,train_129225211,0000a68812bc7e98c42888dfb1c07da0.jpg,94974f937d4c2433,Paper Bag Victoria Secret,249114794
1,train_3386243561,00039780dfc94d01db8676fe789ecd05.jpg,af3f9460c2838f0f,"Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DO...",2937985045
2,train_2288590299,000a190fdd715a2a36faed16e2c65df7.jpg,b94cb00ed3e50f78,Maling TTS Canned Pork Luncheon Meat 397 gr,2395904891
3,train_2406599165,00117e4fc239b1b641ff08340b429633.jpg,8514fc58eafea283,Daster Batik Lengan pendek - Motif Acak / Camp...,4093212188
4,train_3369186413,00136d1cf4edede0203f32f05f660588.jpg,a6f319f924ad708c,Nescafe \xc3\x89clair Latte 220ml,3648931069


## Data preprocessing


In [34]:
UNITS    = {'ml', 'g', 'gr', 'gram', 'kg', 'l', 'cm', 'mm', 'pcs', 'pc', 'm'}
UNIT_MAP = {'gr': 'g', 'gram': 'g', 'pc': 'pcs'}

In [35]:
def fix_bytes_escapes(s):
    # remove the b"..." wrapper, e.g.  b"Tanakan Tablet (30's)"
    if s[:2] in ("b'", 'b"') and s[-1] in ("'", '"'):
        s = s[2:-1]
    # turn literal text like \xc3\x89 into the real character
    try:
        s = s.encode('utf-8').decode('unicode_escape').encode('latin1').decode('utf-8')
    except (UnicodeError, ValueError):
        pass
    return s

In [36]:
def split_num_unit(w):
    # "100ml"->("100", "ml")
    i=0
    while i < len(w) and (w[i].isdigit() or w[i] == '.'):
        i+=1
    return w[:i], w[i:]

In [37]:
def clean(s, noise=frozenset()):
    s = unidecode(fix_bytes_escapes(s)).lower()

    s = ''.join(c if (c.isalnum() or c == '.') else ' ' for c in s)
    tokens = [w.strip('.') for w in s.split()]
    tokens = [w for w in tokens if w]

    out = []
    for w in tokens:
        num, unit = split_num_unit(w)
        if num and unit in UNITS:
            w = num + UNIT_MAP.get(unit, unit)
        elif w in UNITS and out and out[-1].replace('.', '').isdigit():
            w = out.pop() + UNIT_MAP.get(w, w)
        out.append(w)

    return ' '.join(w for w in out if w not in noise)

In [38]:
df['clean'] = df.title.map(clean)
df.head(10)

,posting_id,image,image_phash,title,label_group,clean
0,train_129225211,0000a68812bc7e98c42888dfb1c07da0.jpg,94974f937d4c2433,Paper Bag Victoria Secret,249114794,paper bag victoria secret
1,train_3386243561,00039780dfc94d01db8676fe789ecd05.jpg,af3f9460c2838f0f,"Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DO...",2937985045,double tape 3m vhb 12mm x 4 5m original double...
2,train_2288590299,000a190fdd715a2a36faed16e2c65df7.jpg,b94cb00ed3e50f78,Maling TTS Canned Pork Luncheon Meat 397 gr,2395904891,maling tts canned pork luncheon meat 397g
3,train_2406599165,00117e4fc239b1b641ff08340b429633.jpg,8514fc58eafea283,Daster Batik Lengan pendek - Motif Acak / Camp...,4093212188,daster batik lengan pendek motif acak campur l...
4,train_3369186413,00136d1cf4edede0203f32f05f660588.jpg,a6f319f924ad708c,Nescafe \xc3\x89clair Latte 220ml,3648931069,nescafe eclair latte 220ml
5,train_2464356923,0013e7355ffc5ff8fb1ccad3e42d92fe.jpg,bbd097a7870f4a50,CELANA WANITA (BB 45-84 KG)Harem wanita (bisa...,2660605217,celana wanita bb 45 84kg harem wanita bisa cod
6,train_1802986387,00144a49c56599d45354a1c28104c039.jpg,f815c9bb833ab4c8,Jubah anak size 1-12 thn,1835033137,jubah anak size 1 12 thn
7,train_1806152124,0014f61389cbaa687a58e38a97b6383d.jpg,eea7e1c0c04da33d,KULOT PLISKET SALUR /CANDY PLISKET /WISH KULOT...,1565741687,kulot plisket salur candy plisket wish kulot p...
8,train_86570404,0019a3c6755a194cb2e2c12bfc63972e.jpg,ea9af4f483249972,"[LOGU] Tempelan kulkas magnet angka, tempelan ...",2359912463,logu tempelan kulkas magnet angka tempelan ang...
9,train_831680791,001be52b2beec40ddc1d2d7fc7a68f08.jpg,e1ce953d1a70618f,BIG SALE SEPATU PANTOFEL KULIT KEREN KERJA KAN...,2630990665,big sale sepatu pantofel kulit keren kerja kan...


In [39]:
spread=defaultdict(set)
for text,grp in zip(df.clean,df.label_group):
  for w in set(text.split()):
    spread[w].add(grp)
k=50
top_k=sorted([(w,len(g)) for w,g in spread.items()],key=lambda x:-x[-1])[:k]
print(top_k)

[('original', 1097), ('murah', 1075), ('1', 892), ('wanita', 871), ('anak', 810), ('dan', 791), ('untuk', 707), ('cod', 644), ('2', 608), ('3', 536), ('pria', 523), ('anti', 520), ('isi', 511), ('premium', 508), ('warna', 497), ('termurah', 490), ('motif', 485), ('bahan', 483), ('import', 446), ('bayi', 431), ('promo', 431), ('bisa', 425), ('set', 425), ('polos', 370), ('baju', 367), ('100', 367), ('fashion', 365), ('mini', 361), ('new', 357), ('tas', 353), ('air', 352), ('x', 348), ('free', 335), ('alat', 331), ('masker', 327), ('baby', 327), ('grosir', 318), ('celana', 315), ('4', 313), ('tempat', 313), ('5', 312), ('by', 307), ('hitam', 301), ('in', 298), ('ready', 298), ('1kg', 291), ('6', 283), ('dengan', 276), ('cream', 270), ('tangan', 268)]


In [40]:
noise = {'original', 'murah', 'termurah', 'promo', 'free', 'ready', 'new', 'grosir',
         'import', 'premium', 'cod', 'bisa', 'dan', 'untuk', 'dengan', 'in', 'by'}
df['clean_nn'] = df.title.map(lambda t: clean(t, noise))

In [41]:
def gsplit(d, frac, seed=42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    a, b = next(splitter.split(d, groups=d.label_group))
    return d.iloc[a].reset_index(drop=True), d.iloc[b].reset_index(drop=True)

trainval, test = gsplit(df, 0.20)
train, val     = gsplit(trainval, 0.25)

In [42]:
def make_pairs(d, seed=42, max_pos=100):
    rng = np.random.default_rng(seed)
    labels = d.label_group.values

    # 1) Positives: pairs of listings from the same group
    pos = []
    for group, rows in d.groupby('label_group').indices.items():
        group_pairs = list(itertools.combinations(rows, 2))    # every pair inside the group
        if len(group_pairs) > max_pos:                         # cap big groups
            keep = rng.choice(len(group_pairs), max_pos, replace=False)
            group_pairs = [group_pairs[n] for n in keep]
        pos.extend(group_pairs)

    # 2) Negatives: random pairs from different groups, same count as positives
    neg = []
    while len(neg) < len(pos):
        i, j = rng.integers(0, len(d), 2)
        if labels[i] != labels[j]:
            neg.append((i, j))

    # 3) One table
    rows = [(i, j, 1) for i, j in pos] + [(i, j, 0) for i, j in neg]
    p = pd.DataFrame(rows, columns=['i', 'j', 'y'])
    for c in ['posting_id', 'title', 'clean', 'clean_nn', 'image_phash']:
        p[c + '_a'] = d[c].values[p.i]
        p[c + '_b'] = d[c].values[p.j]
    return p

tr_p, va_p, te_p = make_pairs(train), make_pairs(val), make_pairs(test)
print(tr_p.y.value_counts())

y
1    33824
0    33824
Name: count, dtype: int64


# `BaseLine- TF-IDF and BoW`
### Techniques Used:
- Floor: BoW [Bag of words Embedding]
- Baseline: TF-IDF

- Both the techniques are applied on both `clean` and `clean_nn` which are the cleaned up columns and noise reduced cleaned up colmuns.

In [43]:
def pair_scores(vec,pairs,col):
  A=normalize(vec.transform(pairs[col+'_a']))
  B=normalize(vec.transform(pairs[col+'_b']))
  return np.asarray(A.multiply(B).sum(axis=1)).ravel()

def best_threshold(y, scores):
    p, r, t = precision_recall_curve(y, scores)
    f1 = 2 * p * r / (p + r + 1e-9)
    return t[np.argmax(f1[:-1])]

In [44]:
def run_experiment(name,vec,col="clean"):
  vec.fit(train[col])
  s_val=pair_scores(vec,va_p,col)
  thr=best_threshold(va_p.y,s_val)
  s_te=pair_scores(vec,te_p,col)
  pred = (s_te >= thr).astype(int)
  result = {
        "experiment": name,
        "column": col,
        "threshold": round(float(thr), 3),
        "precision": precision_score(te_p.y, pred),
        "recall":recall_score(te_p.y, pred),
        "f1":f1_score(te_p.y, pred),
        "pr_auc":average_precision_score(te_p.y, s_te),
        "accuracy":metrics.accuracy_score(te_p.y, pred),
        "roc_auc":metrics.roc_auc_score(te_p.y, s_te),
    }
  return result, s_te

In [45]:
RESULTS=[]

BoW_hist,BoW_scores=run_experiment("BoW_counts(+cosine)",CountVectorizer(ngram_range=(1,1)),col="clean")
RESULTS.append(BoW_hist)

TF_IDF_hist,TF_IDF_scores=run_experiment("TF_IDF(+cosine)",TfidfVectorizer(),col="clean")
RESULTS.append(TF_IDF_hist)

BoW_hist_nn,BoW_scores_nn=run_experiment("BoW_counts(+cosine+nn)",CountVectorizer(ngram_range=(1,1)),col="clean_nn")
RESULTS.append(BoW_hist_nn)

TF_IDF_hist_nn,TF_IDF_scores_nn=run_experiment("TF_IDF(+cosine+nn)",TfidfVectorizer(),col="clean_nn")
RESULTS.append(TF_IDF_hist_nn)

base_res_df=pd.DataFrame(RESULTS)
base_res_df

,experiment,column,threshold,precision,recall,f1,pr_auc,accuracy,roc_auc
0,BoW_counts(+cosine),clean,0.124,0.979020,0.957984,0.968388,0.984820,0.968728,0.984400
1,TF_IDF(+cosine),clean,0.093,0.987393,0.957185,0.972054,0.985324,0.972482,0.984825
2,BoW_counts(+cosine+nn),clean_nn,0.123,0.982083,0.958863,0.970334,0.984690,0.970685,0.984385
3,TF_IDF(+cosine+nn),clean_nn,0.093,0.987722,0.957505,0.972379,0.985062,0.972801,0.984687


### Inferences:
- All four perform about the same (F1 around 0.96 to 0.97).
- TF-IDF's precision is consistently a little higher than BoW's, in all three runs.
- Noise removal makes no consistent difference for TF-IDF.

In [46]:
tr_p.to_csv("/content/drive/MyDrive/Colab Notebooks/Shopee_taskB_results/train_pairs.csv", index=False)
va_p.to_csv("/content/drive/MyDrive/Colab Notebooks/Shopee_taskB_results/val_pairs.csv", index=False)
te_p.to_csv("/content/drive/MyDrive/Colab Notebooks/Shopee_taskB_results/test_pairs.csv", index=False)
base_res_df.to_csv("/content/drive/MyDrive/Colab Notebooks/Shopee_taskB_results/base_results.csv", index=False)
print(len(tr_p), len(va_p), len(te_p))

67648 23012 25038


# `Experiment A - TF-IDF with character n-gram`

## TF-IDF with character n_gram

This combines TF-IDF with character n-gram which add a lot of versatile features to TF-IDF.
In this, each word is broken up into subparts which act as the vocabulary.
`char_wb` will be used. It only separates each word and not two or more words together.

### Advantages:
- High Typo Tolerance: Captures overlapping letter sequences. If a user types a misspelling like "tabel" instead of "table", the sub-strings still map closely, preventing the model from breaking down.
- Solves the Out-of-Vocabulary (OOV) Problem: Handles new, rare, or unseen words smoothly. It analyzes the sub-components of a word rather than throwing it out for not being in a pre-defined dictionary.
- This dataset in particular consists of several indo-english words which can be written in various spellings depending on the vendor. Sub-words help navigate this problem. Even if the whole word does not match, some subparts match.

### Disadvantages:
- Vocabulary Explosion: Generates a massive number of unique features. Breaking text down into combinations of 2 to 5 characters creates exceptionally wide, sparse matrices that consume vast amounts of RAM and storage.
- Loss of semantic meanings: This happens as sub-parts of a word cannot truely showcase the meaning of that word always. Sub-parts only show structural patterns.



In [47]:
Results_expA=[]
for lo,hi in [(1,2),(2,3),(2,5)]:
    hist,char_scores=run_experiment(f"TF_IDF_char({lo}-{hi})(+cosine)",TfidfVectorizer(analyzer='char_wb',ngram_range=(lo,hi)),col="clean")
    Results_expA.append(hist)
    hist,char_scores_nn=run_experiment(f"TF_IDF_char({lo}-{hi})(+cosine+nn)",TfidfVectorizer(analyzer='char_wb',ngram_range=(lo,hi)),col="clean_nn")
    Results_expA.append(hist)

In [48]:
Results_expA_df=pd.DataFrame(Results_expA)
Results_expA_df

,experiment,column,threshold,precision,recall,f1,pr_auc,accuracy,roc_auc
0,TF_IDF_char(1-2)(+cosine),clean,0.592,0.961930,0.898155,0.928949,0.978386,0.931304,0.970201
1,TF_IDF_char(1-2)(+cosine+nn),clean_nn,0.573,0.960847,0.905663,0.932440,0.979982,0.934380,0.972203
2,TF_IDF_char(2-3)(+cosine),clean,0.183,0.987574,0.958623,0.972883,0.993304,0.973281,0.990318
3,TF_IDF_char(2-3)(+cosine+nn),clean_nn,0.188,0.989916,0.956626,0.972986,0.993488,0.973440,0.990584
4,TF_IDF_char(2-5)(+cosine),clean,0.151,0.991416,0.959422,0.975156,0.994671,0.975557,0.991982
5,TF_IDF_char(2-5)(+cosine+nn),clean_nn,0.155,0.992556,0.958543,0.975253,0.994811,0.975677,0.992178


In [66]:
Results_expA_df.to_csv("/content/drive/MyDrive/Colab Notebooks/Shopee_taskB_results/EXPA_results.csv", index=False)

### Inferences:
- `TF_IDF_char(1-2)(+cosine)` is a broken setting. The character n-grams technique clearly needs more characters.
- Longer n-grams perform better. Going from 2-3 to 2-5 improves every metric, so bigger chunks (close to whole words but still tolerant of spelling variation) work best.
- `Char 2-5` beats `word TF-IDF` in ranking quality. PR-AUC rises from ~0.985 to ~0.995 and ROC-AUC from ~0.985 to ~0.992.
- The gain is in precision, not recall. Precision rises from ~0.987 to ~0.991-0.993, while recall stays at ~0.958.
- F1 score gain is small.
- Recall almost stays same ~96% for all methods. This suggests that the barrier is in the model rather than the data.
- `clean__nn` makes no visual difference at all.



# Experiment B - Sentence Transformer

## Sentence Transformer
### Model details
- Model name: paraphrase-multilingual-MiniLM-L12-v2
- 12 layer Encoder
- Embedding dimension 384

### Advantages:
- Unlike TF-IDF, this model has been trained on a contrastive objective, so it encodes semantic meanings better.
- It is quite light weight.
- trained on 50+ languages.
- It can capture contextual meanings.

### Disadvantages:
- Its weak against typos.
- Max 128-length sequences are allowed.


### Implementation Details:
- Reduce the 128-sequence length to 64 for faster execution.
-

In [50]:
MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_ID, device="cpu")
model.max_seq_length = 64

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [51]:
def embed_pair_scores(pairs, col):
    texts = pd.unique(pd.concat([pairs[col + "_a"], pairs[col + "_b"]]))   # each title encoded once
    E = model.encode(list(texts), batch_size=64,
                     normalize_embeddings=True, show_progress_bar=True)    # length-1 vectors
    pos = {t: i for i, t in enumerate(texts)}
    A = E[pairs[col + "_a"].map(pos).values]
    B = E[pairs[col + "_b"].map(pos).values]
    return (A * B).sum(axis=1)

In [52]:
emb = {}
Results_expB=[]
for col in ["clean","clean_nn"]:
    t0 = time.time()
    s_val = embed_pair_scores(va_p, col)
    s_te  = embed_pair_scores(te_p, col)
    secs  = time.time() - t0

    thr  = best_threshold(va_p.y, s_val)
    pred = (s_te >= thr).astype(int)
    emb[col] = {"thr": thr, "scores": s_te}

    Results_expB.append({
        "experiment": "MiniLM-L12-multilingual + cosine", "column": col,
        "threshold": round(float(thr), 3),
        "precision": precision_score(te_p.y, pred),
        "recall": recall_score(te_p.y, pred),
        "f1": f1_score(te_p.y, pred),
        "pr_auc": average_precision_score(te_p.y, s_te),
        "encode_seconds": round(secs, 1)
        })


Batches:   0%|          | 0/102 [00:00<?, ?it/s]

Batches:   0%|          | 0/104 [00:00<?, ?it/s]

Batches:   0%|          | 0/101 [00:00<?, ?it/s]

Batches:   0%|          | 0/103 [00:00<?, ?it/s]

In [60]:
Results_expB[0]["accuracy"]=metrics.accuracy_score(te_p.y, (emb["clean"]["scores"] >= Results_expB[0]["threshold"]).astype(int))
Results_expB[0]["roc_auc"]=metrics.roc_auc_score(te_p.y, emb["clean"]["scores"])
Results_expB[1]["accuracy"]=metrics.accuracy_score(te_p.y, (emb["clean_nn"]["scores"] >= Results_expB[1]["threshold"]).astype(int))
Results_expB[1]["roc_auc"]=metrics.roc_auc_score(te_p.y,emb["clean_nn"]["scores"])

In [61]:
Results_expB_df=pd.DataFrame(Results_expB)
Results_expB_df

,experiment,column,threshold,precision,recall,f1,pr_auc,encode_seconds,accuracy,roc_auc
0,MiniLM-L12-multilingual + cosine,clean,0.433,0.952361,0.922997,0.937449,0.984562,326.2,0.938374,0.981021
1,MiniLM-L12-multilingual + cosine,clean_nn,0.430,0.949563,0.927870,0.938591,0.985030,300.0,0.939172,0.981792


In [65]:
Results_expB_df.to_csv("/content/drive/MyDrive/Colab Notebooks/Shopee_taskB_results/EXPB_results.csv",index=False)

## Diagnostics

In [63]:
te_p_diag=te_p.copy()
te_p_diag["char"]=char_scores
te_p_diag["emb"]  = emb["clean_nn"]["scores"]
thr_char=0.151
thr_emb=emb["clean_nn"]["thr"]
pos, neg = te_p_diag[te_p_diag.y == 1], te_p_diag[te_p_diag.y == 0]

print("#total:",len(te_p_diag))
print("#pos:",len(pos))
print("#neg:",len(neg))
print("missed by char, found by emb:", ((pos.char < thr_char) & (pos.emb >= thr_emb)).sum())
print("found by char, missed by emb:", ((pos.char >= thr_char) & (pos.emb < thr_emb)).sum())
print("missed by both:", ((pos.char < thr_char) & (pos.emb < thr_emb)).sum())
print("new false positives from emb:", ((neg.emb >= thr_emb) & (neg.char < thr_char)).sum())

#total: 25038
#pos: 12519
#neg: 12519
missed by char, found by emb: 215
found by char, missed by emb: 611
missed by both: 292
new false positives from emb: 558


### Inferences:

- Almost every metric decreased.
- All these indicate that this model is not the best for this task.
- A likely explanation is that the model tries to encorporate language differences. However, this might not mean that it can encorporate the product details exactly. Two entries such as "Safi Age Defy Gold Water Essence 30ml" and "Safi Age Defy Gold Water Essence 100ml" might look the same to the model.
- Rare words like brand names get blurred in the Embedding which can attribute to the decrease in metrics.


# Final results: Evaluation Table

In [64]:
print("======BASELINE RESULTS========")
display(base_res_df)
print("\n======EXPERIMENT A RESULTS========")
display(Results_expA_df)
print("\n======EXPERIMENT B RESULTS========")
display(Results_expB_df)

======BASELINE RESULTS========


,experiment,column,threshold,precision,recall,f1,pr_auc,accuracy,roc_auc
0,BoW_counts(+cosine),clean,0.124,0.979020,0.957984,0.968388,0.984820,0.968728,0.984400
1,TF_IDF(+cosine),clean,0.093,0.987393,0.957185,0.972054,0.985324,0.972482,0.984825
2,BoW_counts(+cosine+nn),clean_nn,0.123,0.982083,0.958863,0.970334,0.984690,0.970685,0.984385
3,TF_IDF(+cosine+nn),clean_nn,0.093,0.987722,0.957505,0.972379,0.985062,0.972801,0.984687



======EXPERIMENT A RESULTS========


,experiment,column,threshold,precision,recall,f1,pr_auc,accuracy,roc_auc
0,TF_IDF_char(1-2)(+cosine),clean,0.592,0.961930,0.898155,0.928949,0.978386,0.931304,0.970201
1,TF_IDF_char(1-2)(+cosine+nn),clean_nn,0.573,0.960847,0.905663,0.932440,0.979982,0.934380,0.972203
2,TF_IDF_char(2-3)(+cosine),clean,0.183,0.987574,0.958623,0.972883,0.993304,0.973281,0.990318
3,TF_IDF_char(2-3)(+cosine+nn),clean_nn,0.188,0.989916,0.956626,0.972986,0.993488,0.973440,0.990584
4,TF_IDF_char(2-5)(+cosine),clean,0.151,0.991416,0.959422,0.975156,0.994671,0.975557,0.991982
5,TF_IDF_char(2-5)(+cosine+nn),clean_nn,0.155,0.992556,0.958543,0.975253,0.994811,0.975677,0.992178



======EXPERIMENT B RESULTS========


,experiment,column,threshold,precision,recall,f1,pr_auc,encode_seconds,accuracy,roc_auc
0,MiniLM-L12-multilingual + cosine,clean,0.433,0.952361,0.922997,0.937449,0.984562,326.2,0.938374,0.981021
1,MiniLM-L12-multilingual + cosine,clean_nn,0.430,0.949563,0.927870,0.938591,0.985030,300.0,0.939172,0.981792


## Conclusions
Pretrained general-purpose sentence embeddings underperformed lexical methods on this task, because product matching depends on exact identifying tokens (brand, variant, size) that paraphrase-trained embeddings blur.